# CAMEL Playground notebook (SENG 691 Group 3, Phase 2)

Phase 1 built the Playground: two CAMEL agents role-play a task while a person
watches, guides, and instructs them. Phase 2 made that Playground continuous,
supervisable, and reliable, with three enhancements:

1. **Persistent session history.** Sessions are saved to a local SQLite
   database as they run, so a session can be reopened after a restart with its
   task, roles, transcript, guidance, and settings intact.
2. **Progress monitoring and run controls.** A deterministic local monitor
   flags repetition, stalls, drift from the task, and possible completion, and
   the run controls add a configurable turn limit and pause and resume.
3. **Model configuration and error recovery.** A preflight check validates the
   provider, model, and credentials before a run starts, and failed calls are
   classified and retried with backoff when the failure is temporary.

Like the Phase 1 notebook, this one drives the real modules the app uses, and
it runs fully offline against the stub model. No API key is needed. Run the
cells from top to bottom.

Before you run it: this notebook is a companion to the code, not a stand-alone
file. It imports the team's own modules, which live beside it in the `demo`
folder: `session_store.py` (the session store), `progress_monitor.py`,
`model_config.py`, `playground.py`, and `role_playing_demo.py`. Clone the
repository, run `./setup.sh` (or `setup.ps1` on Windows), and open this
notebook from the `demo` folder. A lone copy of the `.ipynb` file, away from
those modules, stops in the setup cell below with an explanation.

In [1]:
# ---- Setup: run once per session. Locates the demo modules and imports them. ----
import sys
import tempfile
import time
from pathlib import Path

_here = Path.cwd()
for _candidate in (_here, _here / "demo", *_here.parents):
    if (_candidate / "session_store.py").exists():
        sys.path.insert(0, str(_candidate))
        DEMO_DIR = _candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the demo folder (session_store.py). This notebook "
        "drives the team's own modules, which live beside it in the demo "
        "folder of the repository: session_store.py, progress_monitor.py, "
        "model_config.py, playground.py, and role_playing_demo.py. Clone "
        "the repository, run ./setup.sh (or setup.ps1 on Windows), and open "
        "demo/phase2_demo.ipynb from that clone.")

from model_config import call_with_retry, classify_error, guidance_for, preflight
from playground import Playground
from progress_monitor import ProgressMonitor
from role_playing_demo import PLATFORMS
from session_store import RESTORED_FIELDS, SessionStore

print("Modules loaded from", DEMO_DIR)
print("Everything below runs offline against the stub model.")

Modules loaded from /home/hatch/workspace/seng691-check/camel/demo
Everything below runs offline against the stub model.


## Enhancement 1: persistent session history

In Phase 1 a session lived only in the app's memory, so a restart lost the
conversation. The session store now writes every session to SQLite as it runs.
The next cell starts a session on the stub model, runs two turns, adds a
guidance note and an instruction, then reopens the same session from the
database in a fresh Playground and compares what came back, field by field.

In [2]:
db_path = str(Path(tempfile.mkdtemp()) / "sessions.db")

pg = Playground(db_path=db_path)
pg.start("Write a small trading bot in Python", "Python Programmer",
         "Stock Trader", None, None, True, True, max_turns=12)
pg.run_turns(2)
pg.guide("Focus on risk checks next")
pg.instruct("Add a function that computes a moving average")

snapshot = {
    "task_original": pg.task_original, "specified_task": pg.specified_task,
    "assistant_role": pg.assistant_role, "user_role": pg.user_role,
    "platform": pg.platform_name, "model_name": pg.model_label,
    "mode": pg.mode, "concise": pg.concise, "turn": pg.turn,
    "status": "in_progress",
}
transcript_before = list(pg.transcript)
print(f"Session {pg.session_id} saved: {len(transcript_before)} transcript entries, "
      f"turn {pg.turn}")

t0 = time.perf_counter()
pg2 = Playground.reopen(pg.session_id, db_path)
restore_seconds = time.perf_counter() - t0

restored = {
    "task_original": pg2.task_original, "specified_task": pg2.specified_task,
    "assistant_role": pg2.assistant_role, "user_role": pg2.user_role,
    "platform": pg2.platform_name, "model_name": pg2.model_label,
    "mode": pg2.mode, "concise": pg2.concise, "turn": pg2.turn,
    "status": "done" if pg2.done else "in_progress",
}
matched = sum(1 for f in RESTORED_FIELDS if restored[f] == snapshot[f])
print(f"Fields restored exactly: {matched}/{len(RESTORED_FIELDS)} "
      f"(Phase 1 baseline: 0/{len(RESTORED_FIELDS)})")
print(f"Transcript restored exactly: {pg2.transcript == transcript_before} "
      f"({len(pg2.transcript)} entries), restore took {restore_seconds:.3f} s")

pg2.run_turns(1)
print(f"The reopened session continues: turn {snapshot['turn']} -> {pg2.turn}")

Session 1 saved: 10 transcript entries, turn 3
Fields restored exactly: 10/10 (Phase 1 baseline: 0/10)
Transcript restored exactly: True (10 entries), restore took 0.002 s
The reopened session continues: turn 3 -> 4


## Enhancement 2: progress monitoring and run controls

The monitor reads the transcript after every turn and applies simple word
overlap rules. It makes no model call, so it costs nothing and its decisions
can be checked by hand. The next cell feeds it a scripted conversation in
which the assistant repeats the same haiku, and prints the flags as the
Narrator lines the app shows.

In [3]:
A, U = "Poet", "Editor"
haiku = "Autumn leaves drift down softly over the quiet garden path"
mon = ProgressMonitor(task_text="Write a haiku about autumn leaves falling")

mon.observe(1, [(U, "Write a haiku about autumn leaves"), (A, haiku)], A, U)
for turn in (2, 3, 4):
    flags = mon.observe(turn, [(U, "Try another version please"), (A, haiku)], A, U)
    for f in flags:
        print(f.line())
print(mon.summary_line())

drift = ProgressMonitor(task_text="Plan a weekend trip to Montreal on a student budget")
drift.observe(1, [(U, "Plan the Montreal weekend trip"),
                  (A, "We can budget the train and a hostel in Montreal")], A, U)
drift.observe(2, [(U, "What about food costs for the weekend"),
                  (A, "Street food keeps the Montreal budget low")], A, U)
for f in drift.observe(3, [(U, "Explain quantum entanglement instead"),
                           (A, "Quantum particles share correlated states instantly")], A, U):
    print(f.line())

Progress check: Poet is repeating an earlier message (100% similar) at turn 2.
Progress check: Editor is repeating an earlier message (100% similar) at turn 3.
Progress check: Poet is repeating an earlier message (100% similar) at turn 3.
Progress check: Editor is repeating an earlier message (100% similar) at turn 4.
Progress check: Poet is repeating an earlier message (100% similar) at turn 4.
Progress check: The last 2 turns added almost no new content. The run looks stalled at turn 4.
Progress flags: repetition x5, stall x1. Latest: The last 2 turns added almost no new content. The run looks stalled at turn 4.
Progress check: Recent exchanges barely mention the task itself (0% of its key words) at turn 3. The agents may be drifting from the goal.


The same controls also bound a run. The next cell starts a session with a
turn limit of 2 and asks for 5 turns, then pauses it and shows that a paused
run makes no progress until it is resumed.

In [4]:
pg3 = Playground(db_path=db_path)
pg3.start("Plan a weekend in Montreal", "Travel Planner", "Student",
          None, None, True, False, max_turns=2)
pg3.run_turns(5)
print(f"Turn limit: run stopped at turn {pg3.turn} of 2 ({pg3.terminated})")

pg3.set_paused(True)
print(f"Paused: run_turns returns {pg3.run_turns(1)}, still at turn {pg3.turn}")
pg3.set_paused(False)
print("Resumed: the run can continue from where it stopped.")

Turn limit: run stopped at turn 2 of 2 (Turn limit reached (2 turns))
Paused: run_turns returns [], still at turn 2
Resumed: the run can continue from where it stopped.


## Enhancement 3: model configuration and error recovery

In Phase 1 a setup mistake, such as a missing key or a mistyped model name,
only surfaced after Start, when the first model call failed mid-run. The
preflight check now validates the setup first. The next cell runs it over five
configurations: the offline stub, a provider with and without a key, a local
provider that needs no key, and an unknown provider.

In [5]:
cases = [
    ("offline stub", preflight(PLATFORMS, None, None, stub=True), True),
    ("openai with no key set", preflight(PLATFORMS, "openai", None, env={}), False),
    ("openai with a key set",
     preflight(PLATFORMS, "openai", None, env={"OPENAI_API_KEY": "sk-test"}), True),
    ("ollama needs no key", preflight(PLATFORMS, "ollama", None, env={}), True),
    ("unknown provider, no model",
     preflight(PLATFORMS, "nosuchprovider", None, env={}), False),
]
for label, result, expected_ok in cases:
    verdict = "ok" if result.ok else "caught before the run"
    detail = "" if result.ok else "; ".join(
        f"{c.name}: {c.detail}" for c in result.checks if not c.ok)
    print(f"{label:28s} -> {verdict}" + (f" ({detail})" if detail else ""))

offline stub                 -> ok
openai with no key set       -> caught before the run (API key: OPENAI_API_KEY is not set. Put it in .env, or tick 'offline stub' to try the UI without a key.)
openai with a key set        -> ok
ollama needs no key          -> ok
unknown provider, no model   -> caught before the run (Provider: 'nosuchprovider' is not in the provider table; it will be used as a custom platform with the model name given.; Model name: No model name given and this provider has no default. Enter a model name.; API key: NOSUCHPROVIDER_API_KEY is not set. Put it in .env, or tick 'offline stub' to try the UI without a key.)


When a call does fail during a run, the error is classified and the user gets
guidance for that category. Temporary failures, such as rate limits, timeouts,
and connection problems, are retried with a growing delay between attempts.
Permanent failures, such as a rejected key, fail at once instead of wasting
retries.

In [6]:
examples = ["429 Too Many Requests", "401 invalid api key",
            "model not found", "request timed out", "connection reset by peer"]
for text in examples:
    category = classify_error(RuntimeError(text))
    print(f"{text:28s} -> {category:16s} {guidance_for(category)}")

429 Too Many Requests        -> rate_limit       The model provider's free-tier limit was reached. Wait a minute and try again, or switch provider under Advanced (free limits: OpenRouter 50 requests/day until $10 of credit is bought, then 1,000/day; Groq about 1,000/day).
401 invalid api key          -> auth             The API key was rejected. Check the key for the chosen provider.
model not found              -> model_not_found  That model name is not available on this provider; clear the Model box to use the provider default.
request timed out            -> timeout          The provider did not answer in time. Try again.
connection reset by peer     -> connection       The provider could not be reached. Check the connection and try again.


In [7]:
def flaky(failures, message):
    state = {"calls": 0}

    def fn():
        state["calls"] += 1
        if state["calls"] <= failures:
            raise RuntimeError(message)
        return "recovered"

    return fn, state


# The sleep is replaced here so the notebook does not wait; the app waits
# 0.5 s, then 1.0 s between attempts.
fn, state = flaky(2, "429 Too Many Requests")
out = call_with_retry(fn, sleep=lambda s: None,
                      on_retry=lambda a, c, e: print(
                          f"attempt {a} failed ({c}); retrying"))
print(f"Rate limited twice, then: {out} after {state['calls']} attempts")

auth_fn, auth_state = flaky(5, "401 invalid api key")
try:
    call_with_retry(auth_fn, sleep=lambda s: None)
except RuntimeError:
    print(f"Rejected key: gave up after {auth_state['calls']} attempt, no retries")

attempt 1 failed (rate_limit); retrying
attempt 2 failed (rate_limit); retrying
Rate limited twice, then: recovered after 3 attempts
Rejected key: gave up after 1 attempt, no retries


## What this demonstrates

Each Phase 2 enhancement, running in the modules the app itself uses:

- Sessions survive a restart. The reopened session above restored every saved
  field and its full transcript exactly, and continued from the turn it had
  reached. Phase 1 restored nothing.
- Progress problems are visible while a run is still going. The monitor
  flagged the repeated haiku as repetition at turn 2 and as a stall at turn 4,
  and flagged the Montreal conversation as drift when it left the task. The
  turn limit stopped a run at its limit, and pause held a run in place.
- Setup mistakes are caught before they cost a run. Preflight passed the
  valid configurations and caught the invalid ones before any model call, and
  retry recovered from simulated rate limits, timeouts, and connection errors
  while failing fast on a rejected key.

The full verification suite is `demo/test_phase2.py` (19 checks) and the
benchmarks behind the report's evidence table are in
`demo/benchmark_phase2.py`. Both run offline, the same way this notebook does.
One honest limit, stated in the report as well: the monitor judges word
overlap, not meaning, so its flags point at possible problems rather than
proving them, and the hosted deployment stores sessions on an ephemeral
filesystem, so they can disappear when the service restarts.